In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import torch
import random
import time

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
# trl==0.11
from trl import AutoModelForCausalLMWithValueHead, AutoModelForSeq2SeqLMWithValueHead, create_reference_model, PPOTrainer, PPOConfig
from tqdm import tqdm

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"[INFO] Using {device} device")

[INFO] Using cuda device


In [3]:
np.random.seed(88)
tqdm.pandas()

# 1. Sentimental Analysis (for reward system)

In [4]:
sentiment_analysis = pipeline('text-classification', 'cardiffnlp/twitter-roberta-base-sentiment-latest')

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [5]:
# Test
result_test = sentiment_analysis("nice job", function_to_apply='none', top_k=None)
result_test

[{'label': 'positive', 'score': 1.9797115325927734},
 {'label': 'neutral', 'score': -0.43869760632514954},
 {'label': 'negative', 'score': -1.8894321918487549}]

# 2. Linguistic Acceptability (for reward system)

In [6]:
cola_tokenizer = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
cola_model = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA")
linguistic_acceptability = pipeline('text-classification', model=cola_model, tokenizer=cola_tokenizer)

Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [7]:
# Test 
result_test = linguistic_acceptability("Her went to a school", function_to_apply='none', top_k=None)
result_test

[{'label': 'LABEL_0', 'score': 1.7046984434127808},
 {'label': 'LABEL_1', 'score': -1.2301501035690308}]

# Wrappers

In [8]:
def neutral_scores(texts):
    scores = []
    results = sentiment_analysis(texts, function_to_apply='none', top_k=None)
    for result in results:
        for label in result:
            if label['label'] == 'neutral':
                scores.append(label['score'])
    return scores
      
neutral_scores(['nice job', 'what a waste', 'hello world!', 'nothing special'])

[-0.43869760632514954,
 -0.01781720295548439,
 0.11776556074619293,
 0.6803277134895325]

In [9]:
def linguistic_acceptable_scores(texts):
    scores = []
    results = linguistic_acceptability(texts, function_to_apply='none', top_k=None)
    for result in results:
        for label in result:
            if label['label'] == 'LABEL_1':
                scores.append(label['score'])
    return scores

linguistic_acceptable_scores(["Her went to a school"])

[-1.2301501035690308]

In [10]:
text_test = ["Donald Trump is facing a widening crisis amid a report claiming that his name appears in US justice department files about Jeffrey Epstein as Congress subpoenas testimony from Epstein accomplice Ghislaine Maxwell."]
print(neutral_scores(text_test), linguistic_acceptable_scores(text_test))

[1.1469523906707764] [1.2529655694961548]


# Dataset preparation

In [11]:
dataset = load_dataset("argilla/news-summary")
dataset['train'][108]

{'text': '(Reuters) - Best known as a New York hedge fund industry executive, Anthony Scaramucci, President Donald Trump’s incoming communications director, has stakes in a film company, a glitzy Manhattan steakhouse and a nutrition business accused by U.S. regulators of making false claims in 2015, financial disclosures show. Overall, Scaramucci has assets in a range of approximately $61 million to $85 million, the forms show. He also has liabilities, such as mortgages and personal loans, of between $6.9 million and $25.8 million.  Scaramucci’s income since the start of 2016 - more than $10 million - is mostly derived from SkyBridge Capital, the hedge fund investment business that he founded in 2005 and is now in the process of selling to join the Trump administration. The disclosure says Scaramucci stands to make more than $50 million from the SkyBridge sale, which he said in May would likely close in June. The deal is on hold pending a regulatory review of its foreign-linked buyers.

In [12]:
flan_t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

In [13]:
dataset = dataset.remove_columns(
    ['prediction', 'prediction_agent', 'annotation', 'annotation_agent', 'metadata', 'status', 'event_timestamp', 'metrics']
)
dataset = dataset.map(
    lambda x: {"input_ids": flan_t5_tokenizer.encode('summarize: ' + x["text"], return_tensors="pt")},
    batched=False,
)
dataset.set_format("pytorch")
dataset['train'][108]

{'text': '(Reuters) - Best known as a New York hedge fund industry executive, Anthony Scaramucci, President Donald Trump’s incoming communications director, has stakes in a film company, a glitzy Manhattan steakhouse and a nutrition business accused by U.S. regulators of making false claims in 2015, financial disclosures show. Overall, Scaramucci has assets in a range of approximately $61 million to $85 million, the forms show. He also has liabilities, such as mortgages and personal loans, of between $6.9 million and $25.8 million.  Scaramucci’s income since the start of 2016 - more than $10 million - is mostly derived from SkyBridge Capital, the hedge fund investment business that he founded in 2005 and is now in the process of selling to join the Trump administration. The disclosure says Scaramucci stands to make more than $50 million from the SkyBridge sale, which he said in May would likely close in June. The deal is on hold pending a regulatory review of its foreign-linked buyers.

# Flan T5

In [14]:
flan_t5_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained("google/flan-t5-small")
flan_t5_model_ref = create_reference_model(flan_t5_model)

# Proximal Policy Optimization

In [15]:
import wandb

ppo_config = PPOConfig(
    model_name="google/flan-t5-small",
    batch_size=8,
    mini_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1.41e-5,
    remove_unused_columns=False,
    log_with="wandb",
    optimize_cuda_cache=True,
)

/home/huo/Workdir/venv/py3transformer/lib/python3.9/site-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


In [20]:
# https://huggingface.co/docs/trl/main/en/how_to_train#what-is-the-concern-with-negative-kl-divergence
generation_kwargs = {
    #"min_length": -1, # don't ignore the EOS token
    "min_length": 3,
    "top_k": 0.0, # no top-k sampling
    "top_p": 1.0, # no nucleus sampling
    "do_sample": True,
    "pad_token_id": flan_t5_tokenizer.eos_token_id,
    #"pad_token_id": flan_t5_tokenizer.pad_token_id,
    #"eos_token_id": flan_t5_tokenizer.eos_token_id,
    "max_new_tokens": 32, # specify how many tokens you want to generate at most
}

In [17]:
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

In [18]:
ppo_trainer = PPOTrainer(
    ppo_config, flan_t5_model, flan_t5_model_ref, flan_t5_tokenizer, dataset['train'], data_collator=collator
)

/home/huo/Workdir/venv/py3transformer/lib/python3.9/site-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(
wandb: Currently logged in as: firelouiszj (firelouiszj-opensee) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [21]:
for epoch in tqdm(range(1)):
    for batch in tqdm(ppo_trainer.dataloader):
        staging_data = dict()
        staging_data["query"] = ['summarize: ' + b for b in batch["text"]]

        # 1. Generate response
        query_tensors = [_.squeeze() for _ in batch["input_ids"]]
        response_tensors = []
        for query in query_tensors:
            response = ppo_trainer.generate(query.squeeze(), **generation_kwargs)
            response_tensors.append(response.squeeze())

        # 2. Calculate reward
        staging_data["response"] = [flan_t5_tokenizer.decode(r.squeeze(), skip_special_tokens=True) for r in response_tensors]
        staging_data['cola_scores'] = linguistic_acceptable_scores(staging_data["response"])
        staging_data['neutral_scores'] = neutral_scores(staging_data["response"])
        tuples = zip(staging_data['cola_scores'], staging_data['neutral_scores'])

        rewards = [1 * values[0] +  0.8 * values[1] for values in tuples]
        rewards = [torch.tensor([_]) for _ in rewards]
        print(rewards)

        # 3. PPO training
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)

        stats['env/reward'] = np.mean([r.cpu().numpy() for r in rewards])
        ppo_trainer.log_stats(stats, staging_data, rewards)

  0%|          | 0/125 [00:00<?, ?it/s]

[tensor([2.8003]), tensor([3.0872]), tensor([2.8696]), tensor([3.3498]), tensor([0.8305]), tensor([2.5145]), tensor([2.5544]), tensor([2.4550])]



  1%|          | 1/125 [00:02<05:55,  2.87s/it]

[tensor([0.2285]), tensor([-0.0987]), tensor([0.6248]), tensor([2.5654]), tensor([0.8533]), tensor([0.9931]), tensor([2.1693]), tensor([2.9722])]



  2%|▏         | 2/125 [00:05<05:01,  2.45s/it]

...


 98%|█████████▊| 123/125 [05:25<00:05,  2.80s/it]

[tensor([2.5323]), tensor([0.1244]), tensor([2.5812]), tensor([3.6924]), tensor([1.9022]), tensor([3.0688]), tensor([3.3295]), tensor([0.8508])]



 99%|█████████▉| 124/125 [05:28<00:02,  2.81s/it]

[tensor([3.0929]), tensor([0.6287]), tensor([1.2672]), tensor([0.7915]), tensor([2.1736]), tensor([-0.1460]), tensor([1.7260]), tensor([2.3068])]



100%|██████████| 1/1 [05:30<00:00, 330.48s/it]


In [22]:
flan_t5_model.save_pretrained("flan-t5-rl")
flan_t5_tokenizer.save_pretrained("flan-t5-rl")

('flan-t5-rl/tokenizer_config.json',
 'flan-t5-rl/special_tokens_map.json',
 'flan-t5-rl/spiece.model',
 'flan-t5-rl/added_tokens.json',
 'flan-t5-rl/tokenizer.json')